In [15]:
import os
import gc
import math
import random
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


In [2]:
import gc

for obj in dir():
    if 'model' in obj.lower() or 'train' in obj.lower() or 'valid' in obj.lower():
        del globals()[obj]

# Сборка мусора
gc.collect()

# Очистка кэша CUDA (если используется GPU)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

GPU memory allocated: 0.00 MB
GPU memory cached: 0.00 MB


In [3]:
# MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

TRAIN_CSV = "./data/train.csv"

OUTPUT_DIR = "./llama_patent_similarity"

MAX_LEN = 384

TRAIN_BATCH_SIZE = 4
VALID_BATCH_SIZE = 4

GRAD_ACCUM_STEPS = 4

EPOCHS = 3

LR = 2e-4
WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_BF16 = torch.cuda.is_bf16_supported()

In [4]:
def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


In [5]:
df = pd.read_csv(TRAIN_CSV)

# Expected columns:
# anchor, target, context, score

print(df.head())

                 id     anchor                  target context  score
0  37d61fd2272659b1  abatement  abatement of pollution     A47   0.50
1  7b9652b17b68b7a4  abatement          act of abating     A47   0.75
2  36d72442aefd8232  abatement         active catalyst     A47   0.25
3  5296b0c19e1ce60e  abatement     eliminating process     A47   0.50
4  54c1e3b9184cb5b6  abatement           forest region     A47   0.00


In [6]:
# Replace with full CPC hierarchy if available
cpc_df = pd.read_csv("./data/archive/titles.csv")


cpc_map = dict(zip(cpc_df["code"], cpc_df["title"]))

def enrich_context(cpc):

    return cpc_map.get(cpc, cpc)

df["context_text"] = df["context"].apply(enrich_context)



In [7]:
def build_prompt(row):

    return f"""
Patent Classification:
{row['context_text']}

Anchor Phrase:
{row['anchor']}

Target Phrase:
{row['target']}

Task:
Predict semantic similarity score between the anchor phrase and target phrase.
""".strip()

df["text"] = df.apply(build_prompt, axis=1)



In [8]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED,
)

train_idx, valid_idx = next(
    splitter.split(
        df,
        groups=df["anchor"]
    )
)

train_df = df.iloc[train_idx].reset_index(drop=True)
valid_df = df.iloc[valid_idx].reset_index(drop=True)

print("\nTrain size:", len(train_df))
print("Valid size:", len(valid_df))



Train size: 27302
Valid size: 9171


In [ ]:
token = "<Your token>"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [10]:
class PatentDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_len
    ):

        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = row["text"]
        score = row["score"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(score, dtype=torch.float)
        }

In [11]:
train_dataset = PatentDataset(
    train_df,
    tokenizer,
    MAX_LEN
)

valid_dataset = PatentDataset(
    valid_df,
    tokenizer,
    MAX_LEN
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=(
        torch.bfloat16
        if USE_BF16
        else torch.float16
    ),

    bnb_4bit_use_double_quant=True,
)

In [13]:
class PatentSimilarityModel(nn.Module):

    def __init__(self, model_name, token=token):

        super().__init__()

        self.backbone = AutoModel.from_pretrained(
            model_name,

            quantization_config=bnb_config,

            device_map="auto",

            output_hidden_states=True,
            
            token=token,

            dtype=(
                torch.bfloat16
                if USE_BF16
                else torch.float16
            )
        )

        hidden_size = self.backbone.config.hidden_size

        # last-4-layer pooling regression head
        self.regressor = nn.Sequential(

            nn.Linear(hidden_size, hidden_size // 2),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(hidden_size // 2, 1)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
    ):

        outputs = self.backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True,
        )

        hidden_states = outputs.hidden_states

        if hidden_states is None:
            raise ValueError("hidden_states is None. Check model config/output flags.")

        # ====================================================
        # LAST 4 LAYERS MEAN POOLING
        # ====================================================

        stacked = torch.stack([
            hidden_states[-1],
            hidden_states[-2],
            hidden_states[-3],
            hidden_states[-4],
        ]).mean(0)

        # ====================================================
        # LAST TOKEN POOLING
        # ====================================================

        seq_lengths = attention_mask.sum(dim=1) - 1

        pooled = stacked[
            torch.arange(stacked.size(0)),
            seq_lengths
        ]

        logits = self.regressor(pooled)

        logits = logits.squeeze(-1)

        return logits

In [ ]:
model = PatentSimilarityModel(MODEL_NAME, token=token)

In [ ]:
model.backbone.config.output_hidden_states = True
model.backbone.config.return_dict = True

In [ ]:
model.backbone = prepare_model_for_kbit_training(
    model.backbone
)
model.backbone.config.use_cache = False 

In [ ]:
lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="FEATURE_EXTRACTION",
)

model.backbone = get_peft_model(
    model.backbone,
    lora_config
)

model.backbone.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,080,711,680 || trainable%: 0.1425


In [ ]:
model = model.to(DEVICE)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
total_steps = (
    len(train_loader)
    * EPOCHS
    // GRAD_ACCUM_STEPS
)

warmup_steps = int(
    total_steps * WARMUP_RATIO
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    optimizer.zero_grad()

    progress_bar = tqdm(loader)

    for step, batch in enumerate(progress_bar):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        labels = batch["labels"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(
            logits,
            labels
        )

        loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad()

        total_loss += loss.item()

        progress_bar.set_description(
            f"train_loss={loss.item():.4f}"
        )

    return total_loss / len(loader)

In [ ]:
def valid_epoch(model, loader):

    model.eval()

    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        progress_bar = tqdm(loader)

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(DEVICE)

            attention_mask = batch[
                "attention_mask"
            ].to(DEVICE)

            labels = batch["labels"].to(DEVICE)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            loss = criterion(
                logits,
                labels
            )

            preds = torch.sigmoid(logits)

            preds = preds.clamp(0, 1)

            total_loss += loss.item()

            all_preds.extend(
                preds.detach().cpu().numpy()
            )

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

    pearson = pearsonr(
        all_preds,
        all_labels
    )[0]

    mse = np.mean(
        (
            np.array(all_preds)
            - np.array(all_labels)
        ) ** 2
    )

    return (
        total_loss / len(loader),
        pearson,
        mse
    )

In [ ]:
best_pearson = -1

for epoch in range(EPOCHS):

    print(f"\n======== EPOCH {epoch+1}/{EPOCHS} ========")

    train_loss = train_epoch(
        model,
        train_loader
    )

    valid_loss, pearson, mse = valid_epoch(
        model,
        valid_loader
    )

    print(f"\nTrain Loss : {train_loss:.4f}")
    print(f"Valid Loss : {valid_loss:.4f}")
    print(f"Pearson    : {pearson:.4f}")
    print(f"MSE        : {mse:.4f}")

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if pearson > best_pearson:

        best_pearson = pearson

        os.makedirs(
            OUTPUT_DIR,
            exist_ok=True
        )

        model.backbone.save_pretrained(
            OUTPUT_DIR
        )

        tokenizer.save_pretrained(
            OUTPUT_DIR
        )

        torch.save(
            model.state_dict(),
            f"{OUTPUT_DIR}/model.pt"
        )

        print("\nBest model saved!")

    gc.collect()

    torch.cuda.empty_cache()


======== EPOCH 1/3 ========


train_loss=0.1251:   8%|▊         | 558/6826 [06:15<1:10:17,  1.49it/s]


KeyboardInterrupt: 

In [ ]:
def predict_similarity(
    context,
    anchor,
    target
):

    model.eval()

    context = enrich_context(context)

    text = f"""
Patent Classification:
{context}

Anchor Phrase:
{anchor}

Target Phrase:
{target}

Task:
Predict semantic similarity score between the anchor phrase and target phrase.
""".strip()

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)

    attention_mask = encoding[
        "attention_mask"
    ].to(DEVICE)

    with torch.no_grad():

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        score = torch.sigmoid(
            logits
        ).item()

    score = max(0.0, min(1.0, score))

    return score

In [ ]:
score = predict_similarity(
    context="H04W",
    anchor="mobile terminal",
    target="cellular device"
)

print("\nPredicted similarity:", score)